### PSD Pipeline
#### Table of Contents:
* [1. Environment Setup](#1-environment-setup)
    * [1.1 Library Imports](#11-library-imports)
    * [1.2 Dataset loading/inspection](#12-dataset-loadinginspection)
* [2. Data Segmentation](#data-segmentation)
* [3. Feature Extraction and Class Distribution](#3-feature-extraction--class-distribution)
    * [3.1 PSD](#31-psd-feature-extraction)
    * [3.2 Class Distribution](#32-class-distribution)
* [4. Classification Schemes](#4-classification-schemes)
    * [4.1 Emotional vs. Neutral](#41-emotional-vs-neutral-remap--class-distribution)
    * [4.2 Positive vs. Negative](#42-positive-vs-negative-remap--class-distribution)
* [5. Model Training](#5-model-training)
    * [5.1 XGBoost](#51-xgboost)
        * [Emotional vs. Neutral](#511-emotional-vs-neutral)
        * [Positive vs. Negative](#512-positive-vs-negative)




### 1. Environment Setup

##### 1.1 Library Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy.signal import welch

import matplotlib.pyplot as plt
import seaborn as sns

import optuna
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import LeaveOneGroupOut, StratifiedKFold, StratifiedGroupKFold
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, roc_auc_score, balanced_accuracy_score, classification_report


/Users/connor/miniconda3/envs/ml/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


##### 1.2 Dataset loading/inspection

In [2]:
dataset_path = Path('DEED')
eeg_dataset = []
subject_ids = []

for file in dataset_path.iterdir():
    mat = loadmat(file)
    eeg = mat['Data']
    fname = file.stem  
    label_part = [part for part in fname.split("_") if part.startswith("E")][0]
    subject_part = [part for part in fname.split("_") if part.startswith("S")][0]
    label = int(label_part[1:])  
    subject_id = subject_part[1:-1]  # last two digits = subject number
    
    eeg_dataset.append((eeg, label))
    subject_ids.append(subject_id)

print(f"Loaded {len(eeg_dataset)} trials.")
print(f"Unique subjects: {len(set(subject_ids))}")
print(f"Subject IDs: {sorted(set(subject_ids))}")
print("Example shapes:", [(arr.shape, lbl) for arr, lbl in eeg_dataset[:3]])

print("\nSample filename → label mapping:")
for file, (_, label) in zip(dataset_path.iterdir(), eeg_dataset[:10]):
    print(f"  {file.stem} → E{label}")

Loaded 533 trials.
Unique subjects: 34
Subject IDs: ['002', '003', '004', '005', '007', '011', '012', '013', '014', '015', '016', '017', '018', '020', '021', '022', '023', '024', '025', '027', '028', '029', '030', '031', '032', '033', '034', '035', '036', '037', '038', '039', '040', '042']
Example shapes: [((6, 290000), 2), ((6, 36000), 2), ((6, 51000), 3)]

Sample filename → label mapping:
  G_S0321_M1_E2_R1_N2_raw_ref → E2
  G_S0213_M3_E2_R7_N2_raw_ref → E2
  G_S0393_M2_E3_R5_REM_raw_ref → E3
  G_S0243_M3_E2_R2_N2_raw_ref → E2
  G_S0311_M3_E5_R2_N2_raw_ref → E5
  G_S0031_M1_E3_R4_nan_raw_ref → E3
  G_S0043_M2_E2_R5_N2_raw_ref → E2
  G_S0072_M1_E0_R11_N1_raw_ref → E0
  G_S0242_M1_E3_R3_W_raw_ref → E3
  G_S0342_M2_E3_R3_N2_raw_ref → E3


### 2. Data Segmentation

Done into 20s windows.

In [3]:
def segmentation(eeg_dataset, subject_ids, window_sec, fs):
    window_size = int(window_sec * fs)
    X = []
    y = []
    groups = []

    for (eeg_array, label), sid in zip(eeg_dataset, subject_ids):
        n_samples = eeg_array.shape[1]
        start = 0
        while start + window_size <= n_samples:
            window = eeg_array[:, start:start + window_size]
            X.append(window)
            y.append(label)
            groups.append(sid)
            start += window_size  

    return X, y, groups

windows, window_labels, window_groups = segmentation(eeg_dataset, subject_ids, 20, 200)
print(f"Total 20 second windows: {len(windows)}")
print(f"Unique subjects: {len(set(window_groups))}")

Total 20 second windows: 7490
Unique subjects: 34


### 3. Feature Extraction & Class Distribution

PSD features are extracted from each 20s window using Welch's method. For each channel, log band power 
and relative band power are computed across the five standard frequency bands (delta, theta, alpha, beta, 
gamma), alongside time-domain mean and variance. Windows are then remapped into two binary classification schemes: Emotional vs. Neutral and Positive vs. Negative.

##### 3.1 PSD Feature Extraction

In [4]:
def extract_psd_features(segmented_windows, labels, fs=200):
    freq_bands = {
        'delta': (0.5, 4),
            'theta': (4, 8),
            'alpha': (8, 12),
            'beta': (12, 30),
            'gamma': (30, 45)
    }
    
    windows = np.array(segmented_windows)
    n_windows, n_channels, n_samples = windows.shape
    
    nperseg = min(512, n_samples)  
    noverlap = nperseg // 2
    
    all_features = []
    
    for ch_idx in range(n_channels):
        ch_data = windows[:, ch_idx, :]
        f, Pxx = welch(ch_data, fs=fs, nperseg=nperseg, noverlap=noverlap, axis=1)
        
        # Band power per band
        ch_band_powers = []
        for band_name, (low, high) in freq_bands.items():
            idx = np.logical_and(f >= low, f <= high)
            band_power = np.trapz(Pxx[:, idx], f[idx], axis=1) 
            ch_band_powers.append(band_power)
        ch_band_powers = np.column_stack(ch_band_powers) 

        # Log band power
        log_band_power = np.log(ch_band_powers + 1e-10)
        all_features.append(log_band_power)

        # Relative band power
        total_power = ch_band_powers.sum(axis=1, keepdims=True)
        relative_band_power = ch_band_powers / (total_power + 1e-10)
        all_features.append(relative_band_power)

        # Time-domain features
        ch_mean = np.mean(ch_data, axis=1, keepdims=True)
        ch_var = np.var(ch_data, axis=1, keepdims=True)
        all_features.append(ch_mean)
        all_features.append(ch_var)

        
    
    
    X = np.hstack(all_features)
    y = np.array(labels)
    
    return X, y

##### 3.2 Class Distribution

In [5]:
X, y = extract_psd_features(windows, window_labels)
print(X.shape)
print(y.shape) 

print("\n=== Total Class Distribution===")
unique, counts = np.unique(y, return_counts=True)
for label, count in zip(unique, counts):
    print(f"E{label}: {count} windows ({count/len(y)*100:.1f}%)")

(7490, 72)
(7490,)

=== Total Class Distribution===
E0: 1294 windows (17.3%)
E1: 305 windows (4.1%)
E2: 1069 windows (14.3%)
E3: 2855 windows (38.1%)
E4: 1698 windows (22.7%)
E5: 269 windows (3.6%)


### 4. Classification Schemes

##### 4.1 Emotional vs. Neutral (remap + class distribution)  
Emotional {E1, E2, E4, E5} vs. Neutral Dream {E3} 

In [6]:
def remap_emotional_neutral(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 3, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [1, 2, 4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_en, mask_en = remap_emotional_neutral(y)
X_en = X[mask_en]
groups_en = np.array(window_groups)[mask_en]

print(f"\n=== Emotional vs. Neutral Class Distribution ===")
print(f"  Total: {len(y_en)}")
print(f"  Neutral (0): {np.sum(y_en == 0)} ({np.sum(y_en == 0)/len(y_en)*100:.1f}%)")
print(f"  Emotional (1): {np.sum(y_en == 1)} ({np.sum(y_en == 1)/len(y_en)*100:.1f}%)")


=== Emotional vs. Neutral Class Distribution ===
  Total: 6196
  Neutral (0): 2855 (46.1%)
  Emotional (1): 3341 (53.9%)


In [7]:
# Subject Wise Class Distribution (EN)
print("=== Subject Wise Class Distribution (EN) ===")
for subject in sorted(set(groups_en)):
    mask = groups_en == subject
    labels = y_en[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (EN) ===
Subject 002: 185 windows, classes: [0 1], counts: [ 18 167]
Subject 003: 400 windows, classes: [0 1], counts: [188 212]
Subject 004: 245 windows, classes: [0 1], counts: [104 141]
Subject 005: 242 windows, classes: [0 1], counts: [155  87]
Subject 007: 511 windows, classes: [0 1], counts: [ 78 433]
Subject 011: 43 windows, classes: [0], counts: [43]
Subject 012: 95 windows, classes: [0 1], counts: [83 12]
Subject 013: 26 windows, classes: [0 1], counts: [12 14]
Subject 014: 56 windows, classes: [0], counts: [56]
Subject 015: 269 windows, classes: [0 1], counts: [122 147]
Subject 016: 213 windows, classes: [0 1], counts: [184  29]
Subject 017: 213 windows, classes: [0 1], counts: [177  36]
Subject 018: 7 windows, classes: [0], counts: [7]
Subject 020: 105 windows, classes: [0 1], counts: [95 10]
Subject 021: 375 windows, classes: [0 1], counts: [162 213]
Subject 022: 207 windows, classes: [0 1], counts: [112  95]
Subject 023: 173 windows, cla

##### 4.2 Positive vs. Negative (remap + class distribution)
Positive {E4, E5} vs. Negative {E1, E2}

In [8]:
def remap_positive_negative(y):
    y = np.array(y)
    keep_mask = np.isin(y, [1, 2, 4, 5])  
    new_y = np.zeros(len(y), dtype=int)
    new_y[np.isin(y, [4, 5])] = 1  
    return new_y[keep_mask], keep_mask

y_pn, mask_pn = remap_positive_negative(y)
X_pn = X[mask_pn]
groups_pn = np.array(window_groups)[mask_pn]

print(f"\n=== Positive vs. Negative Class Distribution ===")
print(f"  Total: {len(y_pn)}")
print(f"  Negative (0): {np.sum(y_pn == 0)} ({np.sum(y_pn == 0)/len(y_pn)*100:.1f}%)")
print(f"  Positive (1): {np.sum(y_pn == 1)} ({np.sum(y_pn == 1)/len(y_pn)*100:.1f}%)")


=== Positive vs. Negative Class Distribution ===
  Total: 3341
  Negative (0): 1374 (41.1%)
  Positive (1): 1967 (58.9%)


In [9]:
# Subject Wise Class Distribution (PN)
print("=== Subject Wise Class Distribution (PN) ===")
for subject in sorted(set(groups_pn)):
    mask = groups_pn == subject
    labels = y_pn[mask]
    unique = np.unique(labels)
    print(f"Subject {subject}: {len(labels)} windows, classes: {unique}, counts: {np.bincount(labels)}")

=== Subject Wise Class Distribution (PN) ===
Subject 002: 167 windows, classes: [0 1], counts: [89 78]
Subject 003: 212 windows, classes: [0 1], counts: [ 65 147]
Subject 004: 141 windows, classes: [0 1], counts: [68 73]
Subject 005: 87 windows, classes: [0 1], counts: [33 54]
Subject 007: 433 windows, classes: [0 1], counts: [206 227]
Subject 012: 12 windows, classes: [0], counts: [12]
Subject 013: 14 windows, classes: [1], counts: [ 0 14]
Subject 015: 147 windows, classes: [0 1], counts: [ 22 125]
Subject 016: 29 windows, classes: [1], counts: [ 0 29]
Subject 017: 36 windows, classes: [0 1], counts: [28  8]
Subject 020: 10 windows, classes: [0], counts: [10]
Subject 021: 213 windows, classes: [0 1], counts: [ 28 185]
Subject 022: 95 windows, classes: [0 1], counts: [46 49]
Subject 023: 100 windows, classes: [0 1], counts: [21 79]
Subject 024: 235 windows, classes: [0 1], counts: [139  96]
Subject 025: 34 windows, classes: [1], counts: [ 0 34]
Subject 027: 26 windows, classes: [0 1], 

### 5. Model Training
For both models and classification schemes, hyperparameters are optimized using Optuna with 5-fold StratifiedGroupKFold on the full dataset. The resulting best parameters are fixed and used for final evaluation with LOSO. For comparison, performance is also assessed using standard 10-fold cross-validation.

##### General Functions for Training

In [10]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def xgb_hyperparameter_training(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_estimators':     trial.suggest_int('n_estimators', 100, 600),
            'max_depth':        trial.suggest_int('max_depth', 3, 8),
            'learning_rate':    trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
            'subsample':        trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'gamma':            trial.suggest_float('gamma', 0, 5),
            'eval_metric':      'logloss',
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            neg = np.sum(y_train == 0)
            pos = np.sum(y_train == 1)
            params['scale_pos_weight'] = neg/pos

            model = XGBClassifier(**params, n_jobs=-1, random_state = 42)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)

            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    best_params['random_state'] = 42
    best_params['n_jobs'] = -1
    return best_params

In [11]:
def xgb_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        #Per fold class weighting
        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)

        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg/pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)

        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [12]:
def xgb_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        neg = np.sum(y_train == 0)
        pos = np.sum(y_train == 1)
        fold_params = params.copy()
        fold_params['scale_pos_weight'] = neg / pos

        model = XGBClassifier(**fold_params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]
        # importance = model.feature_importances_ 
        # print(len(importance))

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

#### 5.1 XGBoost

##### 5.1.1 Emotional vs. Neutral

In [13]:
#Hyperparameter tuning
xgb_en_params = xgb_hyperparameter_training(X_en, y_en, groups_en, "XGBoost", "Emotional vs. Neutral")

[I 2026-03-25 13:08:23,582] A new study created in memory with name: no-name-0766a05c-2384-49fc-812c-c1c5a5817414


=== Hyperparameter Tuning - XGBoost (Emotional vs. Neutral) ===


[I 2026-03-25 13:08:25,467] Trial 0 finished with value: 0.4753947035944825 and parameters: {'n_estimators': 253, 'max_depth': 8, 'learning_rate': 0.1075464866439112, 'subsample': 0.9987649236522196, 'colsample_bytree': 0.8759175841187951, 'min_child_weight': 8, 'gamma': 3.20012915518376}. Best is trial 0 with value: 0.4753947035944825.
[I 2026-03-25 13:08:29,561] Trial 1 finished with value: 0.47328519461471563 and parameters: {'n_estimators': 276, 'max_depth': 5, 'learning_rate': 0.07714698889608224, 'subsample': 0.6877973087544652, 'colsample_bytree': 0.8074570981245253, 'min_child_weight': 5, 'gamma': 0.973691505928248}. Best is trial 0 with value: 0.4753947035944825.
[I 2026-03-25 13:08:33,155] Trial 2 finished with value: 0.48030650489585114 and parameters: {'n_estimators': 440, 'max_depth': 5, 'learning_rate': 0.1716641846758294, 'subsample': 0.8037241321728077, 'colsample_bytree': 0.9400666728394407, 'min_child_weight': 7, 'gamma': 1.960588048791292}. Best is trial 2 with value

Best params: {'n_estimators': 305, 'max_depth': 5, 'learning_rate': 0.23904377290614667, 'subsample': 0.7385452929805647, 'colsample_bytree': 0.8516448502686971, 'min_child_weight': 10, 'gamma': 1.3476536251184512}
Best CV F1: 0.5055


In [14]:
#LOSO Evaluation
xgb_loso = xgb_loso_loop(X_en, y_en, groups_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== LOSO - XGBoost (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.4563 | Accuracy: 0.6000 | F1: 0.4302 | AUROC: 0.4298
Subject 003 | Balanced Accuracy: 0.4532 | Accuracy: 0.4475 | F1: 0.4450 | AUROC: 0.4191
Subject 004 | Balanced Accuracy: 0.4810 | Accuracy: 0.4694 | F1: 0.4694 | AUROC: 0.4562
Subject 005 | Balanced Accuracy: 0.5525 | Accuracy: 0.5785 | F1: 0.5509 | AUROC: 0.5394
Subject 007 | Balanced Accuracy: 0.5040 | Accuracy: 0.5068 | F1: 0.4361 | AUROC: 0.5118
Subject 012 | Balanced Accuracy: 0.6832 | Accuracy: 0.7579 | F1: 0.6140 | AUROC: 0.7560
Subject 013 | Balanced Accuracy: 0.4107 | Accuracy: 0.3846 | F1: 0.3203 | AUROC: 0.4048
Subject 015 | Balanced Accuracy: 0.3906 | Accuracy: 0.3903 | F1: 0.3897 | AUROC: 0.3766
Subject 016 | Balanced Accuracy: 0.4876 | Accuracy: 0.5915 | F1: 0.4571 | AUROC: 0.4818
Subject 017 | Balanced Accuracy: 0.4047 | Accuracy: 0.2864 | F1: 0.2807 | AUROC: 0.3021
Subject 020 | Balanced Accuracy: 0.6158 | Accuracy: 0.4667 | F1: 0.4082 

In [15]:
#10Fold CV
xgb_10f = xgb_ten_fold_cv_loop(X_en, y_en, xgb_en_params, "XGBoost", "Emotional vs. Neutral")


=== 10-Fold CV - XGBoost (Emotional vs. Neutral) ===
Accuracy:          0.6835 ± 0.0131
F1:                0.6808 ± 0.0131
Balanced Accuracy: 0.6806 ± 0.0131
AUROC:             0.7347 ± 0.0139


##### 5.1.2 Positive vs. Negative

In [16]:
#Hyperparameter tuning
xgb_pn_params = xgb_hyperparameter_training(X_pn, y_pn, groups_pn, "XGBoost", "Positive vs. Negative")

[I 2026-03-25 13:11:39,909] A new study created in memory with name: no-name-0ba7e7f4-7ccb-4ebb-852b-853e60f6d7ef


=== Hyperparameter Tuning - XGBoost (Positive vs. Negative) ===


[I 2026-03-25 13:11:42,436] Trial 0 finished with value: 0.5013640150816171 and parameters: {'n_estimators': 266, 'max_depth': 4, 'learning_rate': 0.16358802352977372, 'subsample': 0.6882278294668827, 'colsample_bytree': 0.690729505803738, 'min_child_weight': 5, 'gamma': 0.3540961685714855}. Best is trial 0 with value: 0.5013640150816171.
[I 2026-03-25 13:11:43,875] Trial 1 finished with value: 0.5163628583273392 and parameters: {'n_estimators': 275, 'max_depth': 7, 'learning_rate': 0.1172470084128575, 'subsample': 0.9978063765273572, 'colsample_bytree': 0.9345380242973992, 'min_child_weight': 1, 'gamma': 2.605270125764965}. Best is trial 1 with value: 0.5163628583273392.
[I 2026-03-25 13:11:47,139] Trial 2 finished with value: 0.5269739911772603 and parameters: {'n_estimators': 437, 'max_depth': 4, 'learning_rate': 0.022618047794007678, 'subsample': 0.8543344438394687, 'colsample_bytree': 0.7747508386435529, 'min_child_weight': 10, 'gamma': 3.2495839498355954}. Best is trial 2 with va

Best params: {'n_estimators': 196, 'max_depth': 8, 'learning_rate': 0.09990536427891439, 'subsample': 0.898391961026219, 'colsample_bytree': 0.7214612121857437, 'min_child_weight': 8, 'gamma': 2.967395633147413}
Best CV F1: 0.5418


In [17]:
#LOSO Evaluation
xgb_loso = xgb_loso_loop(X_pn, y_pn, groups_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== LOSO - XGBoost (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.6007 | Accuracy: 0.6048 | F1: 0.6007 | AUROC: 0.6077
Subject 003 | Balanced Accuracy: 0.4179 | Accuracy: 0.5142 | F1: 0.4158 | AUROC: 0.3548
Subject 004 | Balanced Accuracy: 0.5975 | Accuracy: 0.5957 | F1: 0.5954 | AUROC: 0.6662
Subject 005 | Balanced Accuracy: 0.5025 | Accuracy: 0.5287 | F1: 0.5024 | AUROC: 0.5309
Subject 007 | Balanced Accuracy: 0.5045 | Accuracy: 0.5012 | F1: 0.5002 | AUROC: 0.5052
Subject 015 | Balanced Accuracy: 0.5135 | Accuracy: 0.8095 | F1: 0.5091 | AUROC: 0.1971
Subject 017 | Balanced Accuracy: 0.8393 | Accuracy: 0.7500 | F1: 0.7243 | AUROC: 0.8036
Subject 021 | Balanced Accuracy: 0.5244 | Accuracy: 0.7793 | F1: 0.5240 | AUROC: 0.5981
Subject 022 | Balanced Accuracy: 0.5342 | Accuracy: 0.5263 | F1: 0.4995 | AUROC: 0.5736
Subject 023 | Balanced Accuracy: 0.3734 | Accuracy: 0.5900 | F1: 0.3711 | AUROC: 0.3376
Subject 024 | Balanced Accuracy: 0.4142 | Accuracy: 0.4043 | F1: 0.4040 

In [18]:
#10Fold CV
xgb_10f = xgb_ten_fold_cv_loop(X_pn, y_pn, xgb_pn_params, "XGBoost", "Positive vs. Negative")


=== 10-Fold CV - XGBoost (Positive vs. Negative) ===
Accuracy:          0.6797 ± 0.0209
F1:                0.6744 ± 0.0203
Balanced Accuracy: 0.6787 ± 0.0198
AUROC:             0.7464 ± 0.0280


#### KNN

In [19]:
def subject_normalize(X, groups):
    X_norm = X.copy()
    for subj in np.unique(groups):
        mask = groups == subj
        scaler = StandardScaler()
        X_norm[mask] = scaler.fit_transform(X[mask])
    return X_norm

In [20]:
optuna.logging.set_verbosity(optuna.logging.INFO)

def knn_hyperparameter_training(X, y, groups, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedGroupKFold(n_splits=5)
        scores = []
        for train_idx, val_idx in cv.split(X, y, groups):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]
            
            groups_train = groups[train_idx]
            groups_val   = groups[val_idx]

            X_train_norm = subject_normalize(X_train, groups_train)
            X_val_norm   = subject_normalize(X_val, groups_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train_norm, y_train)
            preds = model.predict(X_val_norm)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials = 50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")


    # Save best params
    best_params = study.best_params
    return best_params

In [21]:
def knn_loso_loop(X, y, groups, params, model_name, classification_scheme):
    # LOSO evaluation with fixed params
    print(f"\n=== LOSO - {model_name} ({classification_scheme}) ===")
    logo = LeaveOneGroupOut()
    accs = []
    f1s = []
    aurocs = []
    bal_accs = []

    for fold, (train_idx, test_idx) in enumerate(logo.split(X, y, groups)):
        if len(np.unique(y[test_idx])) < 2:
            continue

        subject = groups[test_idx[0]]
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]

        groups_train = groups[train_idx]
        groups_test  = groups[test_idx]

        X_train_norm = subject_normalize(X_train, groups_train)
        X_test_norm  = subject_normalize(X_test, groups_test)

        model = KNeighborsClassifier(**params)
        model.fit(X_train_norm, y_train)
        preds = model.predict(X_test_norm)
        proba = model.predict_proba(X_test_norm)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))
        print(f"Subject {subject} | Balanced Accuracy: {bal_accs[-1]:.4f} | Accuracy: {accs[-1]:.4f} | F1: {f1s[-1]:.4f} | AUROC: {aurocs[-1]:.4f}")

    print(f"Balanced Accuracy:  {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"Accuracy:  {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:  {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"AUROC:  {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")


In [22]:
def knn_ten_fold_cv_loop(X, y, params, model_name, classification_scheme):
    print(f"\n=== 10-Fold CV - {model_name} ({classification_scheme}) ===")
    # 10 Fold Cross CV with same tuned params
    cv_10fold = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
    accs = []
    f1s = []
    bal_accs = []
    aurocs = []

    for train_idx, test_idx in cv_10fold.split(X, y):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)


        model = KNeighborsClassifier(**params)
        model.fit(X_train, y_train)
        preds = model.predict(X_test)
        proba = model.predict_proba(X_test)[:, 1]

        accs.append(accuracy_score(y_test, preds))
        f1s.append(f1_score(y_test, preds, average='macro'))
        aurocs.append(roc_auc_score(y_test, proba))
        bal_accs.append(balanced_accuracy_score(y_test, preds))

    print(f"Accuracy:          {np.mean(accs):.4f} ± {np.std(accs):.4f}")
    print(f"F1:                {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")
    print(f"Balanced Accuracy: {np.mean(bal_accs):.4f} ± {np.std(bal_accs):.4f}")
    print(f"AUROC:             {np.mean(aurocs):.4f} ± {np.std(aurocs):.4f}")

In [23]:
#Hyperparameter tuning
knn_en_params = knn_hyperparameter_training(X_en, y_en, groups_en, "KNN", "Emotional vs. Neutral")

[I 2026-03-25 13:14:20,871] A new study created in memory with name: no-name-c6c48e08-d2b8-4e12-8a83-c38ba1c8332d


=== Hyperparameter Tuning - KNN (Emotional vs. Neutral) ===


[I 2026-03-25 13:14:24,335] Trial 0 finished with value: 0.49813277395839267 and parameters: {'n_neighbors': 10, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 12}. Best is trial 0 with value: 0.49813277395839267.
[I 2026-03-25 13:14:27,312] Trial 1 finished with value: 0.5019818328331377 and parameters: {'n_neighbors': 1, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 44}. Best is trial 1 with value: 0.5019818328331377.
[I 2026-03-25 13:14:30,630] Trial 2 finished with value: 0.5035470885265683 and parameters: {'n_neighbors': 14, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 32}. Best is trial 2 with value: 0.5035470885265683.
[I 2026-03-25 13:14:31,304] Trial 3 finished with value: 0.48924543057784725 and parameters: {'n_neighbors': 27, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 19}. Best is trial 2 with value: 0.5035470885265683.
[I 2026-03-25 13:14:34,136] Trial 4 finished with value: 0.5019818328331377 and parameters: {'n_neighbors': 1, 'wei

Best params: {'n_neighbors': 17, 'weights': 'distance', 'metric': 'cosine', 'leaf_size': 32}
Best CV F1: 0.5049


In [24]:
#LOSO Evaluation
knn_en_loso = knn_loso_loop(X_en, y_en, groups_en, knn_en_params, "KNN", "Emotional vs. Neutral")


=== LOSO - KNN (Emotional vs. Neutral) ===
Subject 002 | Balanced Accuracy: 0.4413 | Accuracy: 0.5730 | F1: 0.4156 | AUROC: 0.4301
Subject 003 | Balanced Accuracy: 0.4754 | Accuracy: 0.4800 | F1: 0.4742 | AUROC: 0.4477
Subject 004 | Balanced Accuracy: 0.5008 | Accuracy: 0.4980 | F1: 0.4963 | AUROC: 0.5188
Subject 005 | Balanced Accuracy: 0.4980 | Accuracy: 0.4959 | F1: 0.4869 | AUROC: 0.5341
Subject 007 | Balanced Accuracy: 0.4912 | Accuracy: 0.5029 | F1: 0.4298 | AUROC: 0.4821
Subject 012 | Balanced Accuracy: 0.4252 | Accuracy: 0.4316 | F1: 0.3638 | AUROC: 0.3976
Subject 013 | Balanced Accuracy: 0.6190 | Accuracy: 0.6154 | F1: 0.6154 | AUROC: 0.6131
Subject 015 | Balanced Accuracy: 0.4974 | Accuracy: 0.5056 | F1: 0.4966 | AUROC: 0.4813
Subject 016 | Balanced Accuracy: 0.3635 | Accuracy: 0.4272 | F1: 0.3462 | AUROC: 0.3160
Subject 017 | Balanced Accuracy: 0.4442 | Accuracy: 0.3521 | F1: 0.3362 | AUROC: 0.4474
Subject 020 | Balanced Accuracy: 0.6184 | Accuracy: 0.3905 | F1: 0.3598 | AU

In [25]:
#10Fold CV
knn_en_10f = knn_ten_fold_cv_loop(X_en, y_en, knn_en_params, "KNN", "Emotional vs. Neutral")


=== 10-Fold CV - KNN (Emotional vs. Neutral) ===
Accuracy:          0.6249 ± 0.0137
F1:                0.6203 ± 0.0138
Balanced Accuracy: 0.6203 ± 0.0137
AUROC:             0.6726 ± 0.0163


In [26]:
#Hyperparameter tuning
knn_pn_params = knn_hyperparameter_training(X_pn, y_pn, groups_pn, "KNN", "Positive vs. Negative")

[I 2026-03-25 13:16:38,466] A new study created in memory with name: no-name-8ad31135-d2a9-47b5-9940-ad4d9ee5a60b


=== Hyperparameter Tuning - KNN (Positive vs. Negative) ===


[I 2026-03-25 13:16:38,771] Trial 0 finished with value: 0.4836587485934884 and parameters: {'n_neighbors': 26, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 37}. Best is trial 0 with value: 0.4836587485934884.
[I 2026-03-25 13:16:39,049] Trial 1 finished with value: 0.506327910443615 and parameters: {'n_neighbors': 16, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 12}. Best is trial 1 with value: 0.506327910443615.
[I 2026-03-25 13:16:39,300] Trial 2 finished with value: 0.47756176510340176 and parameters: {'n_neighbors': 2, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 45}. Best is trial 1 with value: 0.506327910443615.
[I 2026-03-25 13:16:39,421] Trial 3 finished with value: 0.49552644202617613 and parameters: {'n_neighbors': 16, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 29}. Best is trial 1 with value: 0.506327910443615.
[I 2026-03-25 13:16:39,688] Trial 4 finished with value: 0.5075496521441574 and parameters: {'n_neighbors': 5

Best params: {'n_neighbors': 12, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 17}
Best CV F1: 0.5160


In [27]:
#LOSO Evaluation
knn_pn_loso = knn_loso_loop(X_pn, y_pn, groups_pn, knn_pn_params, "KNN", "Positive vs. Negative")


=== LOSO - KNN (Positive vs. Negative) ===
Subject 002 | Balanced Accuracy: 0.5910 | Accuracy: 0.5868 | F1: 0.5866 | AUROC: 0.6004
Subject 003 | Balanced Accuracy: 0.5079 | Accuracy: 0.5377 | F1: 0.5003 | AUROC: 0.5100
Subject 004 | Balanced Accuracy: 0.5009 | Accuracy: 0.5035 | F1: 0.4993 | AUROC: 0.5154
Subject 005 | Balanced Accuracy: 0.4848 | Accuracy: 0.5287 | F1: 0.4825 | AUROC: 0.5407
Subject 007 | Balanced Accuracy: 0.4809 | Accuracy: 0.4896 | F1: 0.4676 | AUROC: 0.4755
Subject 015 | Balanced Accuracy: 0.5700 | Accuracy: 0.6190 | F1: 0.5114 | AUROC: 0.6409
Subject 017 | Balanced Accuracy: 0.6518 | Accuracy: 0.5278 | F1: 0.5185 | AUROC: 0.4978
Subject 021 | Balanced Accuracy: 0.5387 | Accuracy: 0.4883 | F1: 0.4263 | AUROC: 0.5341
Subject 022 | Balanced Accuracy: 0.5235 | Accuracy: 0.5263 | F1: 0.5210 | AUROC: 0.5417
Subject 023 | Balanced Accuracy: 0.5148 | Accuracy: 0.6200 | F1: 0.5062 | AUROC: 0.4506
Subject 024 | Balanced Accuracy: 0.5709 | Accuracy: 0.5362 | F1: 0.5328 | AU

In [28]:
#10Fold CV
#Most likely worse due to class imbalance
knn_pn_10f = knn_ten_fold_cv_loop(X_pn, y_pn, knn_pn_params, "KNN", "Positive vs. Negative")


=== 10-Fold CV - KNN (Positive vs. Negative) ===
Accuracy:          0.6199 ± 0.0300
F1:                0.6121 ± 0.0305
Balanced Accuracy: 0.6145 ± 0.0312
AUROC:             0.6808 ± 0.0338


##NEW

In [29]:
def knn_hyperparameter_training_10f(X, y, model_name, classification_scheme):
    print(f"=== Hyperparameter Tuning (10-Fold) - {model_name} ({classification_scheme}) ===")

    def objective(trial):
        params = {
            'n_neighbors': trial.suggest_int('n_neighbors', 1, 30),
            'weights': trial.suggest_categorical('weights', ['uniform', 'distance']),
            'metric': trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'cosine']),
            'leaf_size': trial.suggest_int('leaf_size', 10, 50),
        }
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
        scores = []
        for train_idx, val_idx in cv.split(X, y):
            X_train, X_val = X[train_idx], X[val_idx]
            y_train, y_val = y[train_idx], y[val_idx]

            scaler = StandardScaler()
            X_train = scaler.fit_transform(X_train)
            X_val = scaler.transform(X_val)

            model = KNeighborsClassifier(**params)
            model.fit(X_train, y_train)
            preds = model.predict(X_val)
            scores.append(f1_score(y_val, preds, average='macro'))

        return np.mean(scores)

    study = optuna.create_study(direction='maximize')
    study.optimize(objective, n_trials=50)
    print(f"Best params: {study.best_params}")
    print(f"Best CV F1: {study.best_value:.4f}")

    return study.best_params

In [30]:
knn_en_alt_params = knn_hyperparameter_training_10f(X_en, y_en, "knn", "Emotional vs. Neutral")

[I 2026-03-25 13:16:54,115] A new study created in memory with name: no-name-5e690ecd-e542-40b9-a32f-ea9535d4e1e8


=== Hyperparameter Tuning (10-Fold) - knn (Emotional vs. Neutral) ===


[I 2026-03-25 13:16:54,408] Trial 0 finished with value: 0.6182651186221542 and parameters: {'n_neighbors': 20, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 47}. Best is trial 0 with value: 0.6182651186221542.
[I 2026-03-25 13:16:54,512] Trial 1 finished with value: 0.6019555430251222 and parameters: {'n_neighbors': 1, 'weights': 'distance', 'metric': 'euclidean', 'leaf_size': 27}. Best is trial 0 with value: 0.6182651186221542.
[I 2026-03-25 13:16:54,941] Trial 2 finished with value: 0.6456862512895795 and parameters: {'n_neighbors': 6, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 18}. Best is trial 2 with value: 0.6456862512895795.
[I 2026-03-25 13:16:55,198] Trial 3 finished with value: 0.6283817299312517 and parameters: {'n_neighbors': 11, 'weights': 'uniform', 'metric': 'euclidean', 'leaf_size': 14}. Best is trial 2 with value: 0.6456862512895795.
[I 2026-03-25 13:16:55,754] Trial 4 finished with value: 0.6526250418540502 and parameters: {'n_neighbors': 2

Best params: {'n_neighbors': 18, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 30}
Best CV F1: 0.6609


In [31]:
knn_pn_alt_params = knn_hyperparameter_training_10f(X_pn, y_pn, "knn", "Positive vs. Negative")

[I 2026-03-25 13:17:35,204] A new study created in memory with name: no-name-248d8b72-209c-4741-ae68-7ce567e81130


=== Hyperparameter Tuning (10-Fold) - knn (Positive vs. Negative) ===


[I 2026-03-25 13:17:36,185] Trial 0 finished with value: 0.5741362908667895 and parameters: {'n_neighbors': 30, 'weights': 'uniform', 'metric': 'cosine', 'leaf_size': 12}. Best is trial 0 with value: 0.5741362908667895.
[I 2026-03-25 13:17:36,363] Trial 1 finished with value: 0.6003513040036607 and parameters: {'n_neighbors': 28, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 42}. Best is trial 1 with value: 0.6003513040036607.
[I 2026-03-25 13:17:36,541] Trial 2 finished with value: 0.5962317067482976 and parameters: {'n_neighbors': 25, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 33}. Best is trial 1 with value: 0.6003513040036607.
[I 2026-03-25 13:17:36,689] Trial 3 finished with value: 0.6070173399066544 and parameters: {'n_neighbors': 5, 'weights': 'distance', 'metric': 'manhattan', 'leaf_size': 22}. Best is trial 3 with value: 0.6070173399066544.
[I 2026-03-25 13:17:36,817] Trial 4 finished with value: 0.601056050047571 and parameters: {'n_neighbors': 10

Best params: {'n_neighbors': 24, 'weights': 'uniform', 'metric': 'manhattan', 'leaf_size': 10}
Best CV F1: 0.6364


In [36]:
knn_en_10f_alt = knn_ten_fold_cv_loop(X_en, y_en, knn_en_alt_params, "KNN", "Emotional vs. Neutral")


=== 10-Fold CV - KNN (Emotional vs. Neutral) ===
Accuracy:          0.6622 ± 0.0152
F1:                0.6563 ± 0.0157
Balanced Accuracy: 0.6560 ± 0.0155
AUROC:             0.7165 ± 0.0158


In [39]:
knn_pn_10f_alt = knn_ten_fold_cv_loop(X_pn, y_pn, knn_pn_alt_params, "KNN", "Positive vs. Negative")


=== 10-Fold CV - KNN (Positive vs. Negative) ===
Accuracy:          0.6447 ± 0.0269
F1:                0.6343 ± 0.0261
Balanced Accuracy: 0.6351 ± 0.0254
AUROC:             0.7058 ± 0.0296
